# Final Stage F: 51R2 vs 53R2 18-Mask Comparator
## Page-local ordinal alignment

This notebook replaces invalid global call-number alignment.

It does **not** recursively index either capture directory. It reads the two existing lightweight call-trace CSV files, filters to the seven affected pages, sorts calls within each page by their original global call number, assigns a new `page_call_ordinal`, joins 51R2 and 53R2 on `(page, page_call_ordinal)`, and loads only the masks at differing ordinal positions.

Expected exceptional distribution:

- Page003: 2
- Page004: 2
- Page005: 1
- Page008: 1
- Page036: 1
- Page040: 1
- Page049: 1

Total: 9 call pairs, 18 masks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q numpy pandas opencv-python-headless

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import shutil

import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# ------------------------------------------------------------------
# REQUIRED OVERRIDES
# Point these at the lightweight call tables produced by the
# seven-page capture/comparator. These are CSV files, not call folders.
# ------------------------------------------------------------------
TRACE_51_CSV_OVERRIDE = None  # Path('/content/drive/.../51R2/...csv')
TRACE_53_CSV_OVERRIDE = None  # Path('/content/drive/.../53R2/...csv')

# Optional explicit capture roots. If omitted, they are inferred from
# call_dir / mask_path values already stored in the trace CSVs.
CAPTURE_51_ROOT_OVERRIDE = None
CAPTURE_53_ROOT_OVERRIDE = None

# Optional Notebook45 root. Needed only for ownership mapping.
NOTEBOOK45_ROOT_OVERRIDE = None

FAILING_PAGES = [
    'Page003', 'Page004', 'Page005', 'Page008',
    'Page036', 'Page040', 'Page049',
]

EXPECTED_DIFFERENCES = {
    'Page003': 2,
    'Page004': 2,
    'Page005': 1,
    'Page008': 1,
    'Page036': 1,
    'Page040': 1,
    'Page049': 1,
}

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_page_local_ordinal_18_mask_comparator'
OUT = ROOT / 'stage_f_page_local_ordinal_18_mask_comparator' / RUN
TABLES = OUT / 'tables'
IMAGES = OUT / 'images'
AUDIT = OUT / 'audit'
LOCAL = Path('/content/page_local_ordinal_18_masks')

for p in [OUT, TABLES, IMAGES, AUDIT, LOCAL]:
    p.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)

In [ ]:
# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def semantic_hash(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def boolean_hash(array):
    return semantic_hash((np.asarray(array) > 0).astype(np.uint8))


def normalise_page(value):
    text = str(value).strip()
    m = re.search(r'Page\d+', text, flags=re.IGNORECASE)
    if m:
        digits = re.search(r'\d+', m.group()).group()
        return f'Page{int(digits):03d}'
    return text


def extract_call_number(value):
    if pd.isna(value):
        return None
    m = re.search(r'call[_-]?(\d+)', str(value), flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def choose_column(frame, candidates, label, required=True):
    lower = {str(c).lower(): c for c in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    if required:
        raise RuntimeError(
            f'Could not resolve {label}. Columns: {frame.columns.tolist()}'
        )
    return None


def resolve_trace_path(override, side):
    if override is not None:
        path = Path(override)
        require(path.exists(), f'{side} trace CSV does not exist: {path}')
        return path
    raise RuntimeError(
        f'Set TRACE_{side[:2]}_CSV_OVERRIDE to the existing lightweight '
        f'{side} call-trace CSV. This notebook deliberately does not scan '
        'capture directories to discover it.'
    )


def read_trace(path, side):
    frame = pd.read_csv(path, low_memory=False)
    require(len(frame) > 0, f'{side} trace is empty: {path}')

    page_col = choose_column(
        frame,
        ['page', 'page_51', 'page_53', 'context_page', 'page_name', 'page_id'],
        f'{side} page column',
    )
    call_col = choose_column(
        frame,
        ['call_number', 'call', 'call_id', 'call_dir', 'call_path', 'mask_path'],
        f'{side} call-number/path column',
    )
    path_col = choose_column(
        frame,
        ['mask_path', 'call_dir', 'call_path', 'call_directory'],
        f'{side} call path column',
    )
    hash_col = choose_column(
        frame,
        ['mask_boolean_sha256', 'mask_boolean_hash', 'mask_sha256', 'mask_hash'],
        f'{side} Boolean-mask hash column',
    )
    flags_col = choose_column(frame, ['flags'], f'{side} flags column', required=False)

    out = pd.DataFrame({
        'side': side,
        'page': frame[page_col].map(normalise_page),
        'call_number': frame[call_col].map(extract_call_number),
        'recorded_path': frame[path_col].astype(str),
        'mask_boolean_sha256': frame[hash_col].astype(str).str.strip(),
    })

    if flags_col is not None:
        numeric_flags = pd.to_numeric(frame[flags_col], errors='coerce')
        out['flags'] = numeric_flags
        out = out.loc[numeric_flags == int(cv2.INPAINT_TELEA)].copy()
    else:
        out['flags'] = int(cv2.INPAINT_TELEA)

    out = out.loc[out['page'].isin(FAILING_PAGES)].copy()
    require(not out.empty, f'{side}: no rows remain for the seven affected pages.')
    require(out['call_number'].notna().all(), f'{side}: at least one call number could not be parsed.')
    out['call_number'] = out['call_number'].astype(int)

    duplicate = out.duplicated(['page', 'call_number'], keep=False)
    require(
        not duplicate.any(),
        f'{side}: duplicate page/call_number rows:
'
        + out.loc[duplicate, ['page', 'call_number', 'recorded_path']].to_string(index=False)
    )

    out = out.sort_values(['page', 'call_number']).reset_index(drop=True)
    out['page_call_ordinal'] = out.groupby('page').cumcount() + 1
    return out


def resolve_mask_path(recorded_path, call_number, capture_root=None):
    raw = Path(str(recorded_path))

    candidates = []
    if raw.name == 'mask.npy':
        candidates.append(raw)
    else:
        candidates.append(raw / 'mask.npy')

    if capture_root is not None:
        root = Path(capture_root)
        candidates.extend([
            root / 'calls' / f'call_{int(call_number):06d}' / 'mask.npy',
            root / f'call_{int(call_number):06d}' / 'mask.npy',
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0]

In [ ]:
# ------------------------------------------------------------------
# Read lightweight traces and assign page-local ordinals
# ------------------------------------------------------------------

TRACE_51 = resolve_trace_path(TRACE_51_CSV_OVERRIDE, '51R2')
TRACE_53 = resolve_trace_path(TRACE_53_CSV_OVERRIDE, '53R2')

left = read_trace(TRACE_51, '51R2')
right = read_trace(TRACE_53, '53R2')

left.to_csv(AUDIT / '51r2_calls_with_page_local_ordinals.csv', index=False)
right.to_csv(AUDIT / '53r2_calls_with_page_local_ordinals.csv', index=False)

counts = (
    left.groupby('page').size().rename('calls_51')
    .to_frame()
    .join(right.groupby('page').size().rename('calls_53'), how='outer')
    .fillna(0).astype(int).reset_index()
)
counts['count_equal'] = counts['calls_51'] == counts['calls_53']
counts.to_csv(TABLES / 'page_call_counts.csv', index=False)

display(counts)
require(counts['count_equal'].all(), 'Page-level call counts differ. Page-local ordinal pairing is not valid.')

joined = left.merge(
    right,
    on=['page', 'page_call_ordinal'],
    how='outer',
    validate='one_to_one',
    suffixes=('_51', '_53'),
    indicator=True,
)

require((joined['_merge'] == 'both').all(), 'Incomplete page-local ordinal join.')
joined['mask_hash_equal'] = (
    joined['mask_boolean_sha256_51'] == joined['mask_boolean_sha256_53']
)
joined.to_csv(TABLES / 'all_page_local_ordinal_pairs.csv', index=False)

exceptional = joined.loc[~joined['mask_hash_equal']].copy()
exceptional.to_csv(TABLES / 'nine_differing_page_local_ordinals.csv', index=False)

actual_distribution = exceptional.groupby('page').size().to_dict()
print('Differing ordinal distribution:', actual_distribution)
print('Differing ordinal pairs:', len(exceptional))
display(exceptional[[
    'page', 'page_call_ordinal',
    'call_number_51', 'call_number_53',
    'mask_boolean_sha256_51', 'mask_boolean_sha256_53',
]])

require(len(exceptional) == 9, f'Expected 9 differing ordinal pairs, found {len(exceptional)}.')
require(actual_distribution == EXPECTED_DIFFERENCES,
        f'Unexpected distribution. Expected {EXPECTED_DIFFERENCES}, found {actual_distribution}.')

In [ ]:
# ------------------------------------------------------------------
# Resolve only the 18 exceptional mask paths
# ------------------------------------------------------------------

resolved_rows = []
for _, row in exceptional.iterrows():
    for side in ['51', '53']:
        capture_override = CAPTURE_51_ROOT_OVERRIDE if side == '51' else CAPTURE_53_ROOT_OVERRIDE
        path = resolve_mask_path(
            row[f'recorded_path_{side}'],
            int(row[f'call_number_{side}']),
            capture_override,
        )
        resolved_rows.append({
            'side': '51R2' if side == '51' else '53R2',
            'page': row['page'],
            'page_call_ordinal': int(row['page_call_ordinal']),
            'global_call_number': int(row[f'call_number_{side}']),
            'expected_mask_boolean_sha256': row[f'mask_boolean_sha256_{side}'],
            'mask_path': str(path),
            'exists': path.exists(),
        })

resolved = pd.DataFrame(resolved_rows)
resolved.to_csv(TABLES / 'resolved_18_mask_paths.csv', index=False)
display(resolved)

missing = resolved.loc[~resolved['exists']]
require(
    missing.empty,
    'Some of the 18 masks do not exist. No broad search was attempted. '
    'Set CAPTURE_51_ROOT_OVERRIDE or CAPTURE_53_ROOT_OVERRIDE if the trace '
    'stores relative/stale call paths. Missing rows:
' + missing.to_string(index=False)
)

require((resolved.groupby('side').size() == 9).all(), 'Expected 9 masks per side.')

In [ ]:
# ------------------------------------------------------------------
# Stage the 18 masks locally and verify hashes
# ------------------------------------------------------------------

local_rows = []
for _, row in resolved.iterrows():
    source = Path(row['mask_path'])
    destination = LOCAL / (
        f"{row['side']}__{row['page']}__ord_{int(row['page_call_ordinal']):03d}__"
        f"call_{int(row['global_call_number']):06d}.npy"
    )
    shutil.copy2(source, destination)
    mask = np.load(destination, mmap_mode='r', allow_pickle=False)
    actual = boolean_hash(mask)
    require(
        actual == row['expected_mask_boolean_sha256'],
        f'Boolean mask hash mismatch: {source}
Expected {row["expected_mask_boolean_sha256"]}
Actual   {actual}'
    )
    local_rows.append({**row.to_dict(), 'local_mask_path': str(destination), 'actual_mask_boolean_sha256': actual})
    del mask

local = pd.DataFrame(local_rows)
local.to_csv(AUDIT / 'staged_18_masks.csv', index=False)
print('Staged and hash-verified masks:', len(local))

In [ ]:
# ------------------------------------------------------------------
# Compare each exceptional page-local ordinal pair
# ------------------------------------------------------------------

pair_rows = []
coordinate_rows = []

for (page, ordinal), group in local.groupby(['page', 'page_call_ordinal']):
    require(set(group['side']) == {'51R2', '53R2'}, f'{page} ordinal {ordinal}: incomplete pair.')

    p51 = Path(group.loc[group['side'] == '51R2', 'local_mask_path'].iloc[0])
    p53 = Path(group.loc[group['side'] == '53R2', 'local_mask_path'].iloc[0])

    m51_raw = np.load(p51, mmap_mode='r', allow_pickle=False)
    m53_raw = np.load(p53, mmap_mode='r', allow_pickle=False)
    require(m51_raw.shape == m53_raw.shape, f'{page} ordinal {ordinal}: mask shape mismatch.')

    m51 = np.asarray(m51_raw > 0)
    m53 = np.asarray(m53_raw > 0)
    shared = m51 & m53
    only_51 = m51 & ~m53
    only_53 = m53 & ~m51
    xor = only_51 | only_53

    row51 = group.loc[group['side'] == '51R2'].iloc[0]
    row53 = group.loc[group['side'] == '53R2'].iloc[0]

    pair_rows.append({
        'page': page,
        'page_call_ordinal': int(ordinal),
        'global_call_number_51': int(row51['global_call_number']),
        'global_call_number_53': int(row53['global_call_number']),
        'mask_pixels_51': int(m51.sum()),
        'mask_pixels_53': int(m53.sum()),
        'shared_pixels': int(shared.sum()),
        'only_51_pixels': int(only_51.sum()),
        'only_53_pixels': int(only_53.sum()),
        'xor_pixels': int(xor.sum()),
        'mask_51_subset_of_53': bool(not only_51.any()),
        'mask_53_subset_of_51': bool(not only_53.any()),
    })

    pair_dir = IMAGES / f'{page}__ordinal_{int(ordinal):03d}'
    pair_dir.mkdir(parents=True, exist_ok=False)
    for name, mask in [
        ('mask_51.png', m51), ('mask_53.png', m53),
        ('shared.png', shared), ('only_51.png', only_51),
        ('only_53.png', only_53), ('xor.png', xor),
    ]:
        cv2.imwrite(str(pair_dir / name), mask.astype(np.uint8) * 255)

    ys, xs = np.where(xor)
    for y, x in zip(ys, xs):
        coordinate_rows.append({
            'page': page,
            'page_call_ordinal': int(ordinal),
            'global_call_number_51': int(row51['global_call_number']),
            'global_call_number_53': int(row53['global_call_number']),
            'y': int(y),
            'x': int(x),
            'in_51': bool(m51[y, x]),
            'in_53': bool(m53[y, x]),
            'difference_class': 'ONLY_51' if m51[y, x] else 'ONLY_53',
        })

    del m51_raw, m53_raw, m51, m53, shared, only_51, only_53, xor

pairs_df = pd.DataFrame(pair_rows).sort_values(['page', 'page_call_ordinal'])
coords_df = pd.DataFrame(coordinate_rows).sort_values(['page', 'page_call_ordinal', 'y', 'x'])

pairs_df.to_csv(TABLES / 'exceptional_mask_pairs.csv', index=False)
coords_df.to_csv(TABLES / 'exceptional_mask_coordinate_differences.csv', index=False)

display(pairs_df)
print('Total XOR coordinates:', len(coords_df))

In [ ]:
# ------------------------------------------------------------------
# Optional bounded Notebook45 M3 ownership mapping
# No project-wide recursive search is performed.
# ------------------------------------------------------------------

def resolve_notebook45_root():
    if NOTEBOOK45_ROOT_OVERRIDE is not None:
        root = Path(NOTEBOOK45_ROOT_OVERRIDE)
        require(root.exists(), f'Notebook45 root does not exist: {root}')
        return root

    candidates = [
        ROOT / 'observed_print_classifier_v8_6_notebook45',
        ROOT / 'observed_print_classifier_v8_7_notebook45',
        ROOT / 'notebook45',
    ]
    valid = [p for p in candidates if p.exists()]
    require(
        len(valid) == 1,
        'Could not uniquely resolve Notebook45 root from bounded candidates. '
        'Set NOTEBOOK45_ROOT_OVERRIDE. Candidates found: ' + str(valid)
    )
    return valid[0]


def page_m3_files(root, page):
    direct = [root / 'masks' / page, root / page, root / 'pages' / page]
    folders = [p for p in direct if p.exists() and p.is_dir()]
    require(folders, f'{page}: no bounded Notebook45 page-mask folder found below {root}.')
    files = []
    for folder in folders:
        files.extend(folder.glob('*/M3.png'))
        files.extend(folder.glob('*/m3.png'))
    files = sorted(set(files))
    require(files, f'{page}: no regional M3.png files found in bounded page folders.')
    return files

NOTEBOOK45 = resolve_notebook45_root()
print('Notebook45:', NOTEBOOK45)

ownership_rows = []
inventory_rows = []

for page, page_coords in coords_df.groupby('page'):
    files = page_m3_files(NOTEBOOK45, page)
    masks = []
    expected_shape = None

    for path in files:
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        require(image is not None, f'Unreadable M3: {path}')
        mask = image > 0
        if expected_shape is None:
            expected_shape = mask.shape
        require(mask.shape == expected_shape, f'{page}: inconsistent regional M3 shapes.')
        region_id = path.parent.name
        masks.append((region_id, path, mask))
        inventory_rows.append({
            'page': page,
            'region_id': region_id,
            'm3_path': str(path),
            'pixels': int(mask.sum()),
            'array_sha256': semantic_hash(mask.astype(np.uint8)),
        })

    for _, row in page_coords.iterrows():
        y, x = int(row['y']), int(row['x'])
        require(0 <= y < expected_shape[0] and 0 <= x < expected_shape[1],
                f'{page}: coordinate {(y, x)} outside Notebook45 M3 shape {expected_shape}.')
        owners = [region_id for region_id, _, mask in masks if mask[y, x]]
        ownership_rows.append({
            **row.to_dict(),
            'notebook45_owner_count': len(owners),
            'notebook45_owner_region_ids': '|'.join(owners),
            'single_owner': len(owners) == 1,
        })

inventory_df = pd.DataFrame(inventory_rows)
ownership_df = pd.DataFrame(ownership_rows)
inventory_df.to_csv(AUDIT / 'notebook45_regional_m3_inventory.csv', index=False)
ownership_df.to_csv(TABLES / 'exceptional_coordinate_notebook45_ownership.csv', index=False)

ownership_summary = (
    ownership_df.groupby(['page', 'page_call_ordinal', 'difference_class'], as_index=False)
    .agg(
        coordinates=('x', 'size'),
        all_single_owner=('single_owner', 'all'),
        unique_owner_sets=('notebook45_owner_region_ids', 'nunique'),
        owner_sets=('notebook45_owner_region_ids', lambda x: '|'.join(sorted(set(x)))),
    )
)
ownership_summary.to_csv(TABLES / 'exceptional_coordinate_ownership_summary.csv', index=False)
display(ownership_summary)

In [ ]:
# ------------------------------------------------------------------
# Final verdict
# ------------------------------------------------------------------

all_51_subsets = bool(pairs_df['mask_51_subset_of_53'].all())
all_53_subsets = bool(pairs_df['mask_53_subset_of_51'].all())
all_single_owner = bool(ownership_df['single_owner'].all()) if len(ownership_df) else False

if all_51_subsets and not all_53_subsets:
    diagnosis = 'ALL_EXCEPTIONAL_53R2_MASKS_ARE_STRICT_SUPERSETS_OF_51R2'
    correction = (
        'Inspect the regional-mask construction that adds ONLY_53 pixels. '
        'Replicate the 51R2 page-local regional assignment in 53R2.'
    )
elif all_53_subsets and not all_51_subsets:
    diagnosis = 'ALL_EXCEPTIONAL_51R2_MASKS_ARE_STRICT_SUPERSETS_OF_53R2'
    correction = (
        'Inspect the regional-mask construction that removes pixels in 53R2. '
        'Replicate the 51R2 page-local regional assignment in 53R2.'
    )
else:
    diagnosis = 'EXCEPTIONAL_MASKS_HAVE_BIDIRECTIONAL_COORDINATE_DIFFERENCES'
    correction = (
        'Use the coordinate ownership table to identify the first regional '
        'assignment difference. Do not change Telea parameters.'
    )

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': '51R2 versus 53R2 exceptional mask comparison by page-local call ordinal',
    'alignment_key': ['page', 'page_call_ordinal'],
    'global_call_number_alignment_used': False,
    'full_capture_directory_indexed': False,
    'trace_51_csv': str(TRACE_51),
    'trace_53_csv': str(TRACE_53),
    'page_call_counts_equal': bool(counts['count_equal'].all()),
    'exceptional_pairs': int(len(pairs_df)),
    'masks_loaded': int(len(local)),
    'coordinate_differences': int(len(coords_df)),
    'expected_distribution_matched': actual_distribution == EXPECTED_DIFFERENCES,
    'all_51_masks_subset_of_53': all_51_subsets,
    'all_53_masks_subset_of_51': all_53_subsets,
    'all_exceptional_coordinates_single_notebook45_owner': all_single_owner,
    'diagnosis': diagnosis,
    'recommended_first_correction': correction,
    'read_only': True,
}

verdict_path = OUT / 'page_local_ordinal_18_mask_comparator_verdict.json'
verdict_path.write_text(json.dumps(verdict, indent=2))

print(json.dumps(verdict, indent=2))
print('
Key outputs:')
print(' ', TABLES / 'nine_differing_page_local_ordinals.csv')
print(' ', TABLES / 'resolved_18_mask_paths.csv')
print(' ', TABLES / 'exceptional_mask_pairs.csv')
print(' ', TABLES / 'exceptional_mask_coordinate_differences.csv')
print(' ', TABLES / 'exceptional_coordinate_notebook45_ownership.csv')
print(' ', verdict_path)